# Visual Bayesian optimization with BoTorch

We maximize $f(x)=\sin(6\pi x)+0.4x$ on $[0,1]$. The dashed dense curve is visible to us for teaching, but the optimizer receives only the observations it requests. We use a Gaussian process and numerically stable Log Expected Improvement.

In [ ]:
import sys, torch, botorch, gpytorch, matplotlib
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition.analytic import LogExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'botorch': botorch.__version__, 'gpytorch': gpytorch.__version__, 'matplotlib': matplotlib.__version__})

In [ ]:
torch.manual_seed(17)
dtype = torch.double
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
def f(x): return torch.sin(6 * torch.pi * x) + 0.4 * x
train_x = torch.tensor([[0.08], [0.37], [0.72]], dtype=dtype)
train_y = f(train_x)
grid = torch.linspace(0, 1, 600, dtype=dtype).unsqueeze(-1)

In [ ]:
def fit_model(x, y):
    model = SingleTaskGP(x, y, outcome_transform=Standardize(m=1))
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))
    return model

def plot_step(model, x, y, acq, x_next, step):
    model.eval()
    with torch.no_grad():
        p = model.posterior(grid); mu = p.mean.squeeze(); sd = p.variance.sqrt().squeeze()
        av = acq(grid.unsqueeze(-2)).exp().squeeze()
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    a0.plot(grid.squeeze(), f(grid).squeeze(), 'k--', label='true objective (illustration only)')
    a0.plot(grid.squeeze(), mu, label='posterior mean')
    a0.fill_between(grid.squeeze(), mu-2*sd, mu+2*sd, alpha=.2, label='latent mean ± 2 SD')
    a0.scatter(x.squeeze(), y.squeeze(), color='crimson', zorder=3, label='observed')
    a0.axvline(x_next.item(), color='darkorange', label='next x')
    a0.set_ylabel('objective'); a0.legend(loc='best')
    a1.plot(grid.squeeze(), av, color='purple', label='EI (exp of LogEI)')
    a1.axvline(x_next.item(), color='darkorange'); a1.set(xlabel='x', ylabel='acquisition')
    a1.legend(); fig.suptitle(f'BO iteration {step}; best observed={y.max().item():.3f}'); plt.show()

In [ ]:
trace = []
for step in range(1, 7):
    model = fit_model(train_x, train_y)
    acq = LogExpectedImprovement(model, best_f=float(train_y.max()))
    x_next, log_ei = optimize_acqf(acq, bounds=bounds, q=1, num_restarts=12, raw_samples=256)
    plot_step(model, train_x, train_y, acq, x_next.detach(), step)
    y_next = f(x_next.detach())  # the only new objective evaluation
    trace.append({'step': step, 'x_next': x_next.item(), 'y_next': y_next.item(), 'best_before': train_y.max().item()})
    train_x = torch.cat([train_x, x_next.detach()]); train_y = torch.cat([train_y, y_next])
trace

## What is fitted and optimized?

`fit_gpytorch_mll` fits GP hyperparameters from the observed data. `optimize_acqf` then optimizes the cheap acquisition function, not the expensive objective. Calling `f(x_next)` is the single new objective evaluation in each iteration. The final recommendation in this noiseless example is the best evaluated input.